# Eggcase Finder — milestone 1: single-class detector

Trains a small object detector that finds **mermaid's purses** (any species) in a photo,
then exports it as an ONNX file that the Eggcase Finder web app runs on the phone.

**Runs free on Kaggle** (recommended: ~30 GPU hours/week) **or Google Colab**.

1. Kaggle: *Create → New Notebook → File → Import Notebook* → pick this file.
   In the right panel: *Settings → Accelerator → GPU T4 x2* (or P100), and *Internet on*.
   *Add Input → Upload* your `eggcase_dataset.zip` as a Kaggle dataset.
2. Colab: *File → Upload notebook*, then *Runtime → Change runtime type → T4 GPU*.
3. Run the cells top to bottom. Only the **Settings** cell needs editing.

Output: `eggcase_model.zip` containing `model.onnx` + `model.json`. Copy both into the app's `pwa/model/` folder.

> Licence note: Ultralytics (and models trained with it) are AGPL-3.0. Fine for a private prototype,
> and fine for a free app *if the app's source is published under AGPL*. The app is built so this
> model can be swapped for an Apache-licensed detector (YOLOX / RF-DETR) later.

## 1. Settings

In [ ]:
# ---- Edit these -------------------------------------------------------------
DATASET_ZIP = None        # None = auto-find 'eggcase_dataset*.zip' on Kaggle input / Colab upload
CLASS_NAMES = ["eggcase"] # milestone 1: single class. Add "coin" here if you labelled 2p coins (class id 1).
BASE_MODEL  = "yolo11n.pt"  # nano: ~2.6M params, ~10 MB ONNX. "yolo11s.pt" is more accurate but ~3.5x bigger/slower.
IMG_SIZE    = 640         # must match the app (model.json records it). 640 is a good phone default.
EPOCHS      = 150         # early stopping (PATIENCE) usually ends it sooner
PATIENCE    = 30
BATCH       = 16          # drop to 8 if you get CUDA out-of-memory
VAL_FRACTION = 0.18       # only used if your export has no train/val split
SEED        = 42
# -----------------------------------------------------------------------------

## 2. Install and check the GPU

In [ ]:
%pip install -q ultralytics onnx onnxslim onnxruntime
import torch, ultralytics
print("Ultralytics", ultralytics.__version__, "| torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch the runtime to GPU, CPU training is very slow")

## 3. Find and unpack the dataset

In [ ]:
import glob, os, shutil, zipfile
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("/content")
DATA = WORK / "eggcase_dataset"

if DATASET_ZIP is None:
    hits = glob.glob("/kaggle/input/**/eggcase_dataset*.zip", recursive=True) + glob.glob("/content/eggcase_dataset*.zip")
    if not hits and IN_COLAB:
        from google.colab import files
        print("Choose eggcase_dataset.zip to upload...")
        up = files.upload()
        hits = [str(Path("/content") / k) for k in up]
    # Kaggle sometimes auto-extracts uploaded zips: accept a folder too
    if not hits:
        dirs = [d for d in glob.glob("/kaggle/input/**/images", recursive=True)]
        if dirs:
            src = Path(dirs[0]).parent
            print("Using already-extracted dataset at", src)
            shutil.copytree(src, DATA, dirs_exist_ok=True)
    else:
        DATASET_ZIP = hits[0]

if DATASET_ZIP:
    print("Unzipping", DATASET_ZIP)
    tmp = WORK / "_unzip"
    shutil.rmtree(tmp, ignore_errors=True)
    zipfile.ZipFile(DATASET_ZIP).extractall(tmp)
    # find the folder that contains images/ and labels/
    roots = [p.parent for p in tmp.rglob("images") if (p.parent / "labels").exists()]
    assert roots, "Couldn't find images/ and labels/ folders inside the zip - see PHOTO_GUIDE.md for the layout"
    shutil.rmtree(DATA, ignore_errors=True)
    shutil.copytree(roots[0], DATA)

assert (DATA / "images").exists(), f"No dataset at {DATA}"
print("Dataset at", DATA)

### Split into train/val if the export came out as one flat folder

In [ ]:
import random
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def imgs_in(d): return sorted(p for p in Path(d).glob("*") if p.suffix.lower() in IMG_EXT)

if not (DATA / "images" / "train").exists():
    flat = imgs_in(DATA / "images")
    assert flat, "No images found"
    random.Random(SEED).shuffle(flat)
    n_val = max(1, round(len(flat) * VAL_FRACTION))
    for split, items in (("val", flat[:n_val]), ("train", flat[n_val:])):
        (DATA / "images" / split).mkdir(parents=True, exist_ok=True)
        (DATA / "labels" / split).mkdir(parents=True, exist_ok=True)
        for img in items:
            shutil.move(str(img), DATA / "images" / split / img.name)
            lab = DATA / "labels" / (img.stem + ".txt")
            if lab.exists():
                shutil.move(str(lab), DATA / "labels" / split / lab.name)
    print(f"Split {len(flat)} images -> train {len(flat)-n_val}, val {n_val}")
else:
    print("Existing train/val split found - leaving it alone")

## 4. Sanity-check the labels
Catches the usual problems (missing label files, wrong class ids, boxes outside the image) before you burn GPU time.

In [ ]:
from collections import Counter
problems, stats = [], {}
for split in ("train", "val", "test"):
    idir = DATA / "images" / split
    if not idir.exists():
        continue
    imgs = imgs_in(idir)
    n_boxes, n_empty, sizes, cls = 0, 0, [], Counter()
    for img in imgs:
        lab = DATA / "labels" / split / (img.stem + ".txt")
        lines = [l.split() for l in lab.read_text().splitlines() if l.strip()] if lab.exists() else []
        if not lines:
            n_empty += 1
        for parts in lines:
            if len(parts) != 5:
                problems.append(f"{lab.name}: line has {len(parts)} values (expected 5)"); continue
            c, x, y, w, h = int(float(parts[0])), *map(float, parts[1:])
            cls[c] += 1
            if c >= len(CLASS_NAMES): problems.append(f"{lab.name}: class {c} but CLASS_NAMES has {len(CLASS_NAMES)}")
            if not all(0 <= v <= 1.0001 for v in (x, y, w, h)): problems.append(f"{lab.name}: values not normalised 0-1")
            sizes.append(max(w, h)); n_boxes += 1
    stats[split] = len(imgs)
    print(f"{split:5s}: {len(imgs):4d} images | {n_boxes:5d} boxes | {n_empty:3d} with no boxes (negatives) | per class {dict(cls)}")
    if sizes:
        sizes.sort()
        print(f"       box size (fraction of image): min {sizes[0]:.3f}  median {sizes[len(sizes)//2]:.3f}  max {sizes[-1]:.3f}")
print("\n".join(problems[:20]) or "No label problems found")
assert stats.get("train") and stats.get("val"), "Need both train and val images"

### Look at a few labelled images
If boxes are in the wrong place here, the export format is wrong - fix that before training.

In [ ]:
import matplotlib.pyplot as plt, matplotlib.patches as patches
from PIL import Image
sample = random.Random(1).sample(imgs_in(DATA/"images"/"train"), k=min(6, stats["train"]))
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, img in zip(axes.flat, sample):
    im = Image.open(img); W, H = im.size; ax.imshow(im); ax.set_title(img.name, fontsize=8); ax.axis("off")
    lab = DATA/"labels"/"train"/(img.stem+".txt")
    for l in (lab.read_text().splitlines() if lab.exists() else []):
        c, x, y, w, h = map(float, l.split())
        ax.add_patch(patches.Rectangle(((x-w/2)*W, (y-h/2)*H), w*W, h*H, fill=False, lw=2, ec="lime"))
plt.tight_layout(); plt.show()

## 5. Train

In [ ]:
import yaml
data_yaml = WORK / "data.yaml"
cfg = {"path": str(DATA), "train": "images/train", "val": "images/val", "names": {i: n for i, n in enumerate(CLASS_NAMES)}}
if (DATA/"images"/"test").exists(): cfg["test"] = "images/test"
data_yaml.write_text(yaml.safe_dump(cfg))
print(data_yaml.read_text())

from ultralytics import YOLO
model = YOLO(BASE_MODEL)
results = model.train(
    data=str(data_yaml), imgsz=IMG_SIZE, epochs=EPOCHS, patience=PATIENCE, batch=BATCH, seed=SEED,
    project=str(WORK / "runs"), name="eggcase", exist_ok=True,
    # Augmentation tuned for top-down beach photos: eggcases have no "up", so rotate and flip freely.
    degrees=180, flipud=0.5, fliplr=0.5, mosaic=1.0, close_mosaic=15,
    hsv_h=0.02, hsv_s=0.6, hsv_v=0.45, scale=0.6, translate=0.1,
)
BEST = WORK / "runs" / "eggcase" / "weights" / "best.pt"
print("Best weights:", BEST)

## 6. How good is it?
- **Precision**: of the boxes it drew, how many were real eggcases (low = false alarms on seaweed etc.).
- **Recall**: of the real eggcases, how many it found (low = missed cases, so undercounting).
- **mAP50**: overall detection quality. For a first single-class model, 0.8+ on val is a good start.

The count accuracy cell below is the number that matters for this app.

In [ ]:
best = YOLO(str(BEST))
for split in ("val", "test"):
    if split == "test" and "test" not in cfg: continue
    m = best.val(data=str(data_yaml), split=split, imgsz=IMG_SIZE, verbose=False)
    print(f"{split}: precision {m.box.mp:.3f} | recall {m.box.mr:.3f} | mAP50 {m.box.map50:.3f} | mAP50-95 {m.box.map:.3f}")

In [ ]:
# Count accuracy: predicted count vs true count per image, at the app's default threshold.
CONF = 0.35
split = "test" if "test" in cfg else "val"
rows = []
for img in imgs_in(DATA/"images"/split):
    lab = DATA/"labels"/split/(img.stem+".txt")
    true_n = sum(1 for l in lab.read_text().splitlines() if l.strip() and int(float(l.split()[0])) == 0) if lab.exists() else 0
    r = best.predict(str(img), imgsz=IMG_SIZE, conf=CONF, verbose=False)[0]
    pred_n = int((r.boxes.cls == 0).sum())
    rows.append((img.name, true_n, pred_n))
exact = sum(t == p for _, t, p in rows); off1 = sum(abs(t-p) <= 1 for _, t, p in rows)
mae = sum(abs(t-p) for _, t, p in rows) / len(rows)
print(f"{split}: exact count {exact}/{len(rows)} ({exact/len(rows):.0%}) | within ±1 {off1/len(rows):.0%} | mean abs error {mae:.2f}")
print("Worst images (fix these by adding similar photos):")
for name, t, p in sorted(rows, key=lambda r: -abs(r[1]-r[2]))[:8]:
    print(f"  {name}: true {t}, predicted {p}")

In [ ]:
# Eyeball predictions on a few images
sample = imgs_in(DATA/"images"/split)[:6]
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, img in zip(axes.flat, sample):
    r = best.predict(str(img), imgsz=IMG_SIZE, conf=CONF, verbose=False)[0]
    ax.imshow(r.plot()[..., ::-1]); ax.set_title(f"{img.name}: {len(r.boxes)} found", fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

## 7. Export for the web app
Exports a fixed-size ONNX model (no built-in NMS; the app does NMS itself, which keeps the model portable across runtimes), then checks it gives the same detections as PyTorch.

In [ ]:
import json
onnx_path = Path(best.export(format="onnx", imgsz=IMG_SIZE, opset=17, simplify=True, dynamic=False, nms=False, half=False))
EXPORT = WORK / "eggcase_model"
shutil.rmtree(EXPORT, ignore_errors=True); EXPORT.mkdir()
shutil.copy(onnx_path, EXPORT / "model.onnx")

meta = {
    "name": "eggcase-detector",
    "version": f"m1-{__import__('datetime').date.today().isoformat()}",
    "decoder": "yolo-v8",          # output layout [1, 4 + n_classes, n_anchors]; see pwa/js/decoders.js
    "inputSize": IMG_SIZE,
    "classes": CLASS_NAMES,
    "countClasses": [0],           # which class ids count as eggcases (e.g. not 'coin')
    "defaultConfidence": 0.35,
    "iouThreshold": 0.5,
    "letterboxColor": 114,
    "trainedFrom": BASE_MODEL,
    "licence": "AGPL-3.0 (Ultralytics-trained)",
}
(EXPORT / "model.json").write_text(json.dumps(meta, indent=2))
print("Model size: %.1f MB" % ((EXPORT/"model.onnx").stat().st_size / 1e6))

In [ ]:
# Parity check: run the ONNX file with onnxruntime using the SAME preprocessing the app uses
# (letterbox to square, grey 114 padding, RGB, /255, NCHW) and compare counts with PyTorch.
import numpy as np, onnxruntime as ort

def letterbox(im, size):
    w, h = im.size; s = min(size / w, size / h)
    nw, nh = round(w * s), round(h * s)
    canvas = Image.new("RGB", (size, size), (114, 114, 114))
    px, py = (size - nw) // 2, (size - nh) // 2
    canvas.paste(im.convert("RGB").resize((nw, nh), Image.BILINEAR), (px, py))
    return np.asarray(canvas, dtype=np.float32).transpose(2, 0, 1)[None] / 255.0

def nms(b, s, iou):
    order, keep = s.argsort()[::-1], []
    while order.size:
        i = order[0]; keep.append(i)
        xx1 = np.maximum(b[i,0], b[order[1:],0]); yy1 = np.maximum(b[i,1], b[order[1:],1])
        xx2 = np.minimum(b[i,2], b[order[1:],2]); yy2 = np.minimum(b[i,3], b[order[1:],3])
        inter = np.clip(xx2-xx1, 0, None) * np.clip(yy2-yy1, 0, None)
        area = lambda k: (b[k,2]-b[k,0]) * (b[k,3]-b[k,1])
        order = order[1:][inter / (area(i) + area(order[1:]) - inter + 1e-9) <= iou]
    return keep

sess = ort.InferenceSession(str(EXPORT/"model.onnx"))
mism = 0
for img in imgs_in(DATA/"images"/split)[:25]:
    out = sess.run(None, {sess.get_inputs()[0].name: letterbox(Image.open(img), IMG_SIZE)})[0][0]  # [4+nc, N]
    scores = out[4:].max(0); cls_id = out[4:].argmax(0); m = scores >= CONF
    cx, cy, w, h = out[:4, m]; boxes = np.stack([cx-w/2, cy-h/2, cx+w/2, cy+h/2], 1)
    keep = nms(boxes + cls_id[m][:, None] * 10000.0, scores[m], meta["iouThreshold"])  # offset = per-class NMS, like the app
    n_onnx = int(sum(cls_id[m][k] in meta["countClasses"] for k in keep))
    n_pt = int((best.predict(str(img), imgsz=IMG_SIZE, conf=CONF, iou=meta["iouThreshold"], verbose=False)[0].boxes.cls == 0).sum())
    mism += n_onnx != n_pt
    if n_onnx != n_pt: print(f"  {img.name}: onnx {n_onnx} vs torch {n_pt}")
print("ONNX/PyTorch count mismatches:", mism, "(0-1 is fine; small resize differences can flip borderline boxes)")

## 8. Download

In [ ]:
zip_path = shutil.make_archive(str(WORK / "eggcase_model"), "zip", EXPORT)
print("Created", zip_path)
if IN_COLAB:
    from google.colab import files; files.download(zip_path)
else:
    print("Kaggle: open the Output panel on the right (or /kaggle/working) and download eggcase_model.zip")

## Next
1. Unzip into the app: `pwa/model/model.onnx` and `pwa/model/model.json`.
2. Push to GitHub Pages and open it on your phone (see README).
3. Note what it gets wrong on a real beach, photograph more of that, label, retrain.
   To retrain from where you left off, set `BASE_MODEL` to your previous `best.pt` (upload it as an input).